# Silver — Hospital Data Transformation

**Notebook:** `silver_hospital_data_transformation.ipynb`

Reads the AIHW JSON files from Bronze and prepares the observation and reference tables used by the Gold model.

**Input:** `Files/bronze/data_items/` and `Files/bronze/metadata/`  
**Output:** Silver Delta tables in the Lakehouse  
**Observation grain:** One source-reported measure observation, including its dataset, reported-measure and reporting-unit context.

The notebook standardises fields, keeps suppression and caveat information, adds dataset reporting dates, checks for missing keys and reviews possible duplicates. Duplicate candidates are reported, not removed automatically. The metadata catalogs are also written as Silver reference tables.

This is the cleaned/staging layer before the Gold dimensional model; it is not the Kimball star schema itself.


In [1]:
# Imports, paths and helper for selecting the latest metadata snapshot.

from pyspark.sql import functions as F
from pyspark.sql import types as T

spark.conf.set("spark.sql.session.timeZone", "UTC")

BRONZE_PATH = "Files/bronze/data_items/*.json"
METADATA_PATH = "Files/bronze/metadata"
OBSERVATIONS_TABLE = "silver_hospital_measure_observations"
AUDIT_TABLE = "silver_measure_ingestion_audit"

# Use the latest timestamped metadata snapshot, not every snapshot in the folder.
def latest_metadata_file(prefix):
    files = [
        f for f in notebookutils.fs.ls(METADATA_PATH)
        if f.name.startswith(prefix) and f.name.endswith(".json")
    ]
    if not files:
        raise FileNotFoundError(f"No {prefix}*.json file found in {METADATA_PATH}")
    return sorted(files, key=lambda f: f.name)[-1].path

print("Bronze data path:", BRONZE_PATH)
print("Metadata path:", METADATA_PATH)


StatementMeta(, 937a3a7a-2c7a-4d4f-8be8-2f30a207ba3a, 3, Finished, Available, Finished, False)

Bronze data path: Files/bronze/data_items/*.json
Metadata path: Files/bronze/metadata


In [2]:
# Read each Bronze response and create a file-level audit, including empty responses.

raw_df = (
    spark.read.option("multiLine", True).json(BRONZE_PATH)
    .withColumn("_source_file", F.input_file_name())
    .withColumn("source_file_name", F.regexp_extract(F.col("_source_file"), r"([^/]+)$", 1))
    .withColumn("source_measure_code", F.regexp_extract(F.col("source_file_name"), r"^(MYH\d+)\.json$", 1))
)

if "result" not in raw_df.columns:
    raise ValueError("Bronze JSON is missing the expected 'result' array.")

if raw_df.filter(F.col("source_measure_code") == "").limit(1).count():
    display(raw_df.filter(F.col("source_measure_code") == "").select("source_file_name", "_source_file"))
    raise ValueError("Could not extract measure codes from all Bronze filenames.")

# Read version details when the API envelope provides them.
version_fields = set()
if "version_information" in raw_df.columns and isinstance(raw_df.schema["version_information"].dataType, T.StructType):
    version_fields = set(raw_df.schema["version_information"].dataType.fieldNames())

def version_value(field_name):
    return F.col(f"version_information.{field_name}").cast("string") if field_name in version_fields else F.lit(None).cast("string")

audit = raw_df.select(
    "source_measure_code",
    "source_file_name",
    F.col("_source_file").alias("source_file_path"),
    F.size("result").cast("long").alias("result_record_count"),
    F.when(F.col("result").isNull(), F.lit("MISSING_RESULT"))
     .when(F.size("result") == 0, F.lit("EMPTY_RESULT"))
     .otherwise(F.lit("HAS_DATA")).alias("response_status"),
    version_value("api_version").alias("api_version"),
    version_value("data_version").alias("data_version"),
    version_value("date_uploaded").alias("date_uploaded"),
    version_value("requested_time_stamp").alias("requested_time_stamp"),
    F.current_timestamp().alias("audited_at_utc")
)

print("Bronze response files:", audit.count())
display(audit.orderBy("source_measure_code"))


StatementMeta(, 937a3a7a-2c7a-4d4f-8be8-2f30a207ba3a, 4, Finished, Available, Finished, False)

Bronze response files: 33


SynapseWidget(Synapse.DataFrame, f5b2952f-620f-4f00-a9ef-d922138380c2)

In [3]:
# Check that Bronze contains a file for every measure in the latest catalog.

measures_path = latest_metadata_file("measures_")
measures_envelope = spark.read.option("multiLine", True).json(measures_path)

if "result" not in measures_envelope.columns:
    raise ValueError(f"Measure catalog has no 'result' array: {measures_path}")

catalog_fields = set(measures_envelope.schema["result"].dataType.elementType.fieldNames())
if "measure_code" not in catalog_fields:
    raise ValueError(f"Catalog has no measure_code field: {sorted(catalog_fields)}")

expected_codes = (
    measures_envelope.select(F.explode_outer("result").alias("m"))
    .select(F.upper(F.trim(F.col("m.measure_code").cast("string"))).alias("source_measure_code"))
    .filter(F.col("source_measure_code").isNotNull() & (F.col("source_measure_code") != ""))
    .distinct()
)
actual_codes = audit.select(F.upper(F.trim("source_measure_code")).alias("source_measure_code")).distinct()
missing_codes = expected_codes.join(actual_codes, "source_measure_code", "left_anti")
unexpected_codes = actual_codes.join(expected_codes, "source_measure_code", "left_anti")

print("Catalog measures:", expected_codes.count())
print("Bronze files:", audit.count())
print("Missing Bronze files:", missing_codes.count())
print("Bronze codes not in catalog:", unexpected_codes.count())
if missing_codes.limit(1).count():
    display(missing_codes)
if unexpected_codes.limit(1).count():
    display(unexpected_codes)
if missing_codes.limit(1).count() or unexpected_codes.limit(1).count():
    raise ValueError("Bronze measure coverage does not match the latest measure catalog.")


StatementMeta(, 937a3a7a-2c7a-4d4f-8be8-2f30a207ba3a, 5, Finished, Available, Finished, False)

Catalog measures: 33
Bronze files: 33
Missing Bronze files: 0
Bronze codes not in catalog: 0


In [4]:
# Turn API result records into observation rows and standardise the source fields.

observations_df = (
    raw_df.select("source_measure_code", "_source_file", "result")
    .select("source_measure_code", "_source_file", F.explode_outer("result").alias("record"))
    .filter(F.col("record").isNotNull())
    .select("source_measure_code", "_source_file", "record.*")
)

required_columns = ["data_set_id", "measure_code", "reported_measure_code", "value", "reporting_unit_summary"]
missing_columns = sorted(set(required_columns) - set(observations_df.columns))
if missing_columns:
    raise ValueError(f"Bronze observations are missing required fields: {missing_columns}")

# Standardise key numeric fields and preserve nulls from the source.
for column_name in ["value", "lower_value", "upper_value"]:
    if column_name in observations_df.columns:
        observations_df = observations_df.withColumn(column_name, F.col(column_name).cast("double"))
if "group_number" in observations_df.columns:
    observations_df = observations_df.withColumn("group_number", F.col("group_number").cast("integer"))
observations_df = observations_df.withColumn("data_set_id", F.col("data_set_id").cast("long"))

# Make reporting-unit details usable as normal columns; keep the source struct too.
summary_specs = [
    ("reporting_unit_summary", "reporting_unit"),
    ("peer_group_summary", "peer_group"),
    ("proxy_reporting_unit_summary", "proxy_reporting_unit"),
]
for struct_name, prefix in summary_specs:
    if struct_name in observations_df.columns and isinstance(observations_df.schema[struct_name].dataType, T.StructType):
        for nested_name in observations_df.schema[struct_name].dataType.fieldNames():
            # Keep existing API names readable, without overwriting a column already present.
            if struct_name == "reporting_unit_summary" and nested_name.startswith("reporting_unit_"):
                flat_name = nested_name
            elif struct_name == "peer_group_summary" and nested_name.startswith("peer_group_"):
                flat_name = nested_name
            elif struct_name == "proxy_reporting_unit_summary" and nested_name.startswith("proxy_reporting_unit_"):
                flat_name = nested_name
            else:
                flat_name = f"{prefix}_{nested_name}"
            if flat_name not in observations_df.columns:
                observations_df = observations_df.withColumn(flat_name, F.col(f"{struct_name}.{nested_name}"))

# Keep the source arrays and add small, query-friendly flags/counts.
if "suppressions" in observations_df.columns:
    observations_df = observations_df.withColumn("suppression_count", F.coalesce(F.size("suppressions"), F.lit(0)).cast("integer"))
else:
    observations_df = observations_df.withColumn("suppression_count", F.lit(0).cast("integer"))
if "caveats" in observations_df.columns:
    observations_df = observations_df.withColumn("caveat_count", F.coalesce(F.size("caveats"), F.lit(0)).cast("integer"))
else:
    observations_df = observations_df.withColumn("caveat_count", F.lit(0).cast("integer"))

observations_df = (
    observations_df
    .withColumn("is_suppressed", F.col("suppression_count") > 0)
    .withColumn("is_value_missing", F.col("value").isNull())
    .withColumn("silver_processed_at_utc", F.current_timestamp())
)

# The source code in the filename should agree with the code on the observation.
measure_mismatch = observations_df.filter(
    F.col("measure_code").isNotNull() &
    (F.upper(F.trim("source_measure_code")) != F.upper(F.trim("measure_code")))
)
if measure_mismatch.limit(1).count():
    display(measure_mismatch.select("source_measure_code", "measure_code", "data_set_id").limit(20))
    raise ValueError("Source measure code does not match measure_code on one or more observations.")

bronze_record_count = audit.agg(F.sum("result_record_count").alias("n")).first()["n"] or 0
exploded_record_count = observations_df.count()
print(f"Bronze result records: {bronze_record_count:,}")
print(f"Exploded observation rows: {exploded_record_count:,}")
if bronze_record_count != exploded_record_count:
    raise ValueError("Bronze-to-Silver row reconciliation failed.")


StatementMeta(, 937a3a7a-2c7a-4d4f-8be8-2f30a207ba3a, 6, Finished, Available, Finished, False)

Bronze result records: 1,709,311
Exploded observation rows: 1,709,311


In [5]:
# Add dataset names and reporting dates without multiplying observation rows.

datasets_path = latest_metadata_file("datasets_")
datasets_envelope = spark.read.option("multiLine", True).json(datasets_path)
if "result" not in datasets_envelope.columns:
    raise ValueError(f"Dataset catalog has no 'result' array: {datasets_path}")

dataset_records = datasets_envelope.select(F.explode_outer("result").alias("d"))
dataset_fields = set(dataset_records.schema["d"].dataType.fieldNames())
if "data_set_id" not in dataset_fields:
    raise ValueError(f"Dataset catalog has no data_set_id field: {sorted(dataset_fields)}")

dataset_columns = [F.col("d.data_set_id").cast("long").alias("data_set_id")]
for field_name in ["data_set_name", "reporting_start_date", "reporting_end_date"]:
    if field_name in dataset_fields:
        dataset_columns.append(F.col(f"d.{field_name}").cast("string").alias(field_name))
datasets_df = dataset_records.select(*dataset_columns).filter(F.col("data_set_id").isNotNull())

duplicate_dataset_ids = datasets_df.groupBy("data_set_id").count().filter(F.col("count") > 1)
if duplicate_dataset_ids.limit(1).count():
    display(duplicate_dataset_ids.limit(20))
    raise ValueError("Dataset catalog has duplicate data_set_id values; the join would multiply rows.")

before_join = observations_df.count()
silver_df = observations_df.join(datasets_df, on="data_set_id", how="left")
after_join = silver_df.count()
if before_join != after_join:
    raise ValueError(f"Dataset join changed row count from {before_join:,} to {after_join:,}.")

# Convert reporting dates when the API provides them as strings.
for date_col in ["reporting_start_date", "reporting_end_date"]:
    if date_col in silver_df.columns:
        silver_df = silver_df.withColumn(date_col, F.to_date(F.col(date_col)))

unmatched_datasets = silver_df.filter(F.col("data_set_name").isNull()).count() if "data_set_name" in silver_df.columns else None
print("Dataset metadata file:", datasets_path)
print("Rows before/after dataset join:", before_join, "/", after_join)
if unmatched_datasets is not None:
    print("Rows without a dataset metadata match:", f"{unmatched_datasets:,}")


StatementMeta(, 937a3a7a-2c7a-4d4f-8be8-2f30a207ba3a, 7, Finished, Available, Finished, False)

Dataset metadata file: abfss://2ffe3afa-ed13-4f5d-be42-bd0cad06dcb7@onelake.dfs.fabric.microsoft.com/adf4914a-af3e-4321-8500-8547cc278e1e/Files/bronze/metadata/datasets_20261102T020244Z.json
Rows before/after dataset join: 1709311 / 1709311
Rows without a dataset metadata match: 0


In [6]:
# Check required fields and review duplicate candidates without dropping records.

row_count = silver_df.count()
missing_source_codes = silver_df.filter(F.col("source_measure_code").isNull() | (F.trim("source_measure_code") == "")).count()
missing_measure_codes = silver_df.filter(F.col("measure_code").isNull() | (F.trim("measure_code") == "")).count()
null_values = silver_df.filter(F.col("value").isNull()).count()
suppressed_rows = silver_df.filter(F.col("is_suppressed")).count()

print(f"Silver observations: {row_count:,}")
print(f"Missing source measure codes: {missing_source_codes:,}")
print(f"Missing measure codes: {missing_measure_codes:,}")
print(f"Null values retained: {null_values:,}")
print(f"Rows with suppression details: {suppressed_rows:,}")

if row_count == 0:
    raise ValueError("Silver produced no observations.")
if missing_source_codes or missing_measure_codes:
    raise ValueError("Required measure codes are missing from Silver.")

# A zero count means no rows share this candidate key; it does not justify dropping data.
# Reporting-unit identity is included when available. Keep all observations unchanged.
duplicate_key = [
    c for c in ["source_measure_code", "data_set_id", "measure_code", "reported_measure_code", "group_number"]
    if c in silver_df.columns
]
if "reporting_unit_code" in silver_df.columns:
    duplicate_key.append("reporting_unit_code")

duplicate_candidates = silver_df.groupBy(*duplicate_key).count().filter(F.col("count") > 1)
duplicate_group_count = duplicate_candidates.count()
print(f"Duplicate candidate groups (review only): {duplicate_group_count:,}")
if duplicate_group_count:
    display(duplicate_candidates.orderBy(F.desc("count")).limit(20))
else:
    print("No duplicate candidate groups found for the selected key; no deduplication was applied.")


StatementMeta(, 937a3a7a-2c7a-4d4f-8be8-2f30a207ba3a, 8, Finished, Available, Finished, False)

Silver observations: 1,709,311
Missing source measure codes: 0
Missing measure codes: 0
Null values retained: 655,835
Rows with suppression details: 659,093
Duplicate candidate groups (review only): 0
No duplicate candidate groups found for the selected key; no deduplication was applied.


In [7]:
# Load the latest API metadata catalogs as Silver reference tables.

metadata_tables = {
    "categories_": "silver_measure_categories",
    "measures_": "silver_measures_ref",
    "datasets_": "silver_datasets",
    "reporting_units_": "silver_reporting_units",
    "reporting_unit_types_": "silver_reporting_unit_types",
}

metadata_dfs = {}
for prefix, table_name in metadata_tables.items():
    metadata_path = latest_metadata_file(prefix)
    envelope = spark.read.option("multiLine", True).json(metadata_path)
    if "result" not in envelope.columns:
        raise ValueError(f"Metadata file has no 'result' array: {metadata_path}")
    records = (
        envelope.select(F.explode_outer("result").alias("record"))
        .filter(F.col("record").isNotNull())
        .select("record.*")
    )
    if records.limit(1).count() == 0:
        raise ValueError(f"Metadata catalog is unexpectedly empty: {metadata_path}")
    metadata_dfs[table_name] = records
    print(f"{table_name}: {records.count():,} rows | source: {metadata_path}")


StatementMeta(, 937a3a7a-2c7a-4d4f-8be8-2f30a207ba3a, 9, Finished, Available, Finished, False)

silver_measure_categories: 12 rows | source: abfss://2ffe3afa-ed13-4f5d-be42-bd0cad06dcb7@onelake.dfs.fabric.microsoft.com/adf4914a-af3e-4321-8500-8547cc278e1e/Files/bronze/metadata/categories_20261102T020244Z.json
silver_measures_ref: 33 rows | source: abfss://2ffe3afa-ed13-4f5d-be42-bd0cad06dcb7@onelake.dfs.fabric.microsoft.com/adf4914a-af3e-4321-8500-8547cc278e1e/Files/bronze/metadata/measures_20261102T020244Z.json
silver_datasets: 7,244 rows | source: abfss://2ffe3afa-ed13-4f5d-be42-bd0cad06dcb7@onelake.dfs.fabric.microsoft.com/adf4914a-af3e-4321-8500-8547cc278e1e/Files/bronze/metadata/datasets_20261102T020244Z.json
silver_reporting_units: 1,428 rows | source: abfss://2ffe3afa-ed13-4f5d-be42-bd0cad06dcb7@onelake.dfs.fabric.microsoft.com/adf4914a-af3e-4321-8500-8547cc278e1e/Files/bronze/metadata/reporting_units_20261102T020244Z.json
silver_reporting_unit_types: 6 rows | source: abfss://2ffe3afa-ed13-4f5d-be42-bd0cad06dcb7@onelake.dfs.fabric.microsoft.com/adf4914a-af3e-4321-8500-8547

In [8]:
# Write the Silver tables and compare persisted row counts with the prepared data.

# Write the observation table and file-level audit.
(silver_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(OBSERVATIONS_TABLE))
(audit.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(AUDIT_TABLE))

# Write each flattened metadata catalog to its Silver reference table.
for table_name, dataframe in metadata_dfs.items():
    (dataframe.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name))

persisted_rows = spark.table(OBSERVATIONS_TABLE).count()
persisted_audit_rows = spark.table(AUDIT_TABLE).count()
print(f"Expected observation rows: {row_count:,} | persisted: {persisted_rows:,}")
print(f"Expected audit rows: {audit.count():,} | persisted: {persisted_audit_rows:,}")

if persisted_rows != row_count:
    raise ValueError("Silver observation row-count reconciliation failed.")
if persisted_audit_rows != audit.count():
    raise ValueError("Silver audit row-count reconciliation failed.")

for table_name, dataframe in metadata_dfs.items():
    persisted_count = spark.table(table_name).count()
    expected_count = dataframe.count()
    print(f"{table_name}: expected {expected_count:,} | persisted {persisted_count:,}")
    if persisted_count != expected_count:
        raise ValueError(f"Row-count reconciliation failed for {table_name}.")

print("Silver transformation completed successfully.")


StatementMeta(, 937a3a7a-2c7a-4d4f-8be8-2f30a207ba3a, 10, Finished, Available, Finished, False)

Expected observation rows: 1,709,311 | persisted: 1,709,311
Expected audit rows: 33 | persisted: 33
silver_measure_categories: expected 12 | persisted 12
silver_measures_ref: expected 33 | persisted 33
silver_datasets: expected 7,244 | persisted 7,244
silver_reporting_units: expected 1,428 | persisted 1,428
silver_reporting_unit_types: expected 6 | persisted 6
Silver transformation completed successfully.


In [9]:
# Review file coverage, observation counts and suppression/caveat information.

# File audit summary: empty responses should be visible here, not as fake observation rows.
display(
    spark.table(AUDIT_TABLE)
    .groupBy("response_status")
    .agg(
        F.count("*").alias("measure_files"),
        F.sum("result_record_count").alias("observation_records")
    )
    .orderBy("response_status")
)

# Observation coverage and suppression summary by source measure.
display(
    spark.table(OBSERVATIONS_TABLE)
    .groupBy("source_measure_code")
    .agg(
        F.count("*").alias("observation_count"),
        F.sum(F.when(F.col("value").isNull(), 1).otherwise(0)).alias("null_value_count"),
        F.sum("suppression_count").alias("suppression_entries"),
        F.sum("caveat_count").alias("caveat_entries")
    )
    .orderBy("source_measure_code")
)


StatementMeta(, 937a3a7a-2c7a-4d4f-8be8-2f30a207ba3a, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5d3729d0-9ff8-4c07-9d47-f1936429f506)

SynapseWidget(Synapse.DataFrame, a85d66bf-8671-4ba3-b035-d0267e6adfe4)